# 04 Padé 有理分式逼近

Taylor 多项式使用多项式描述函数在展开点附近的局部行为。Padé 逼近改用有理函数：

$$
R_{m,n}(x)=\frac{P_m(x)}{Q_n(x)},\qquad Q_n(0)=1.
$$

它要求

$$
Q_n(x)f(x)-P_m(x)=O(x^{m+n+1}).
$$

因此 Padé 逼近可以用 Taylor 系数构造，但最终得到的是分式形式。


In [ ]:
import pathlib
import sys

import matplotlib.pyplot as plt
import numpy as np

plt.rcParams["font.sans-serif"] = [
    "Arial Unicode MS",
    "PingFang SC",
    "Heiti SC",
    "SimHei",
    "Noto Sans CJK SC",
    "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False

ROOT = pathlib.Path.cwd()
while not (ROOT / "src" / "py_sc").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import math

from py_sc import pade_eval, pade_from_taylor, polynomial_eval


## 系数方程

设

$$
f(x)=c_0+c_1x+c_2x^2+\cdots,
$$

并令

$$
Q_n(x)=1+q_1x+\cdots+q_nx^n.
$$

条件

$$
Q_n(x)f(x)-P_m(x)=O(x^{m+n+1})
$$

意味着 $x^{m+1},\dots,x^{m+n}$ 的系数必须为零。这给出关于 $q_1,\dots,q_n$ 的线性方程。求出 $Q_n$ 后，$P_m$ 的低阶系数由乘积 $Q_nf$ 的前 $m+1$ 项得到。


In [ ]:
def teaching_pade_from_taylor(c, m, n):
    c = np.asarray(c, dtype=float)
    q = np.zeros(n + 1)
    q[0] = 1.0
    if n > 0:
        A = np.empty((n, n))
        b = np.empty(n)
        for row in range(n):
            power = m + 1 + row
            b[row] = -c[power]
            for col in range(n):
                A[row, col] = c[power - (col + 1)]
        q[1:] = np.linalg.solve(A, b)

    p = np.zeros(m + 1)
    for power in range(m + 1):
        p[power] = sum(q[j] * c[power - j] for j in range(min(power, n) + 1))
    return p, q

exp_coefficients = np.array([1 / math.factorial(k) for k in range(7)], dtype=float)
teaching_p, teaching_q = teaching_pade_from_taylor(exp_coefficients, 3, 3)
formal_p, formal_q = pade_from_taylor(exp_coefficients, 3, 3)
print("教学版和正式版分子最大差异:", np.max(np.abs(teaching_p - formal_p)))
print("教学版和正式版分母最大差异:", np.max(np.abs(teaching_q - formal_q)))


## Taylor 与 Padé 对比

对指数函数，三次 Taylor 多项式为

$$
1+x+\frac{x^2}{2}+\frac{x^3}{6}.
$$

Padé $[3/3]$ 使用同样数量级的 Taylor 信息，但把一部分自由度放到分母上。


In [ ]:
x_eval = np.linspace(-2.0, 2.0, 600)
y_true = np.exp(x_eval)
y_taylor = polynomial_eval(exp_coefficients[:4], x_eval)
y_pade = pade_eval(formal_p, formal_q, x_eval)

print(f"三次 Taylor 最大误差: {np.max(np.abs(y_true - y_taylor)):.3e}")
print(f"Padé [3/3] 最大误差: {np.max(np.abs(y_true - y_pade)):.3e}")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(x_eval, y_true, label="exp(x)")
axes[0].plot(x_eval, y_taylor, "--", label="三次 Taylor")
axes[0].plot(x_eval, y_pade, ":", label="Padé [3/3]")
axes[0].set_title("函数值对比")
axes[0].set_xlabel("x")
axes[0].set_ylabel("函数值")
axes[0].legend()

axes[1].semilogy(x_eval, np.abs(y_true - y_taylor) + 1e-16, label="Taylor 误差")
axes[1].semilogy(x_eval, np.abs(y_true - y_pade) + 1e-16, label="Padé 误差")
axes[1].set_title("误差对比")
axes[1].set_xlabel("x")
axes[1].set_ylabel("绝对误差")
axes[1].legend()
fig.tight_layout()
plt.show()


## Padé 的适用场景

Padé 逼近适合下列情况：

* Taylor 多项式收敛半径受附近奇点限制；
* 函数存在极点或近似极点结构；
* 希望用较低阶模型描述远离展开点的行为；
* 需要在微分方程、传递函数或特殊函数中保留分式结构。

但是 Padé 也可能引入虚假极点，因此需要检查分母零点和逼近区间。


## 小结

* Padé 逼近由 Taylor 系数构造，但结果是有理函数。
* 分母系数来自高阶系数消去条件。
* 分子系数来自低阶 Taylor 系数匹配。
* 对某些函数，Padé 在较宽区间上比同阶 Taylor 多项式更稳健。
